# 04 Causal Inference

Estimate the causal effect of price-increase exposure on customer churn using the synthetic customer dataset.

In [23]:
from math import sqrt
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

try:
    from dowhy import CausalModel
    DOWHY_AVAILABLE = True
except ModuleNotFoundError:
    CausalModel = None
    DOWHY_AVAILABLE = False

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "raw" / "customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CUSTOMERS_PATH = PROJECT_ROOT / "data" / "raw" / "customers.csv"

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:,.4f}".format)

DOWHY_AVAILABLE

False

## Causal Graph / DAG

Treatment: price-increase exposure  
Outcome: customer churn

```mermaid
flowchart LR
    Region[customer_region] --> Treatment[price_increase_exposure]
    Region --> Churn[churn]
    Channel[acquisition_channel] --> Treatment
    Channel --> Churn
    Frequency[purchase_frequency] --> Treatment
    Frequency --> Churn
    Tenure[customer_tenure] --> Treatment
    Tenure --> Churn
    Spending[prior_spending] --> Treatment
    Spending --> Churn
    Treatment --> Discount[discount_usage]
    Discount --> Churn
    Treatment --> Churn
```

For the primary total-effect estimate, discount usage is not included in the main adjustment set because it may sit partly after treatment. A discount-adjusted sensitivity estimate is shown separately.

In [24]:
dag_edges = pd.DataFrame(
    [
        ("customer_region", "price_increase_exposure"),
        ("customer_region", "churn"),
        ("acquisition_channel", "price_increase_exposure"),
        ("acquisition_channel", "churn"),
        ("purchase_frequency", "price_increase_exposure"),
        ("purchase_frequency", "churn"),
        ("customer_tenure", "price_increase_exposure"),
        ("customer_tenure", "churn"),
        ("prior_spending", "price_increase_exposure"),
        ("prior_spending", "churn"),
        ("price_increase_exposure", "discount_usage"),
        ("discount_usage", "churn"),
        ("price_increase_exposure", "churn"),
    ],
    columns=["from", "to"],
)
display(dag_edges)

,from,to
0,customer_region,price_increase_exposure
1,customer_region,churn
2,acquisition_channel,price_increase_exposure
3,acquisition_channel,churn
4,purchase_frequency,price_increase_exposure
5,purchase_frequency,churn
6,customer_tenure,price_increase_exposure
7,customer_tenure,churn
8,prior_spending,price_increase_exposure
9,prior_spending,churn


## Load Synthetic Customer Data

In [25]:
customers = pd.read_csv(
    CUSTOMERS_PATH,
    parse_dates=["first_purchase_date", "last_transaction_date"],
)

def normalize_boolean(series):
    """Normalize CSV boolean values while preserving invalid values as missing."""
    if pd.api.types.is_bool_dtype(series):
        return series
    return series.astype(str).str.lower().map({"true": True, "false": False})

for column in ["price_increase_occurred", "churned"]:
    customers[column] = normalize_boolean(customers[column])

customers["treatment"] = customers["price_increase_occurred"].astype(int)
customers["outcome"] = customers["churned"].astype(int)
customers["discount_user"] = (customers["average_discount_percent"] > 0).astype(int)

primary_confounders = [
    "purchase_frequency",
    "prior_spending",
    "customer_tenure_days",
    "customer_region",
    "acquisition_channel",
]
discount_sensitivity_confounders = primary_confounders + [
    "average_discount_percent",
    "discount_user",
]

display(customers.head())
display(customers[["treatment", "outcome", *primary_confounders, "average_discount_percent", "discount_user"]].isna().sum().to_frame("missing_values"))

,customer_id,first_purchase_date,customer_region,acquisition_channel,customer_tenure_days,purchase_frequency,prior_spending,transaction_count,total_spending,average_discount_percent,price_increase_occurred,churned,last_transaction_date,treatment,outcome,discount_user
0,C000001,2025-10-16,uk,direct,76,8.0000,103.9800,2,103.9800,0.0000,False,True,2025-11-08,0,1,0
1,C000002,2025-07-12,uk,influencer,172,4.2470,120.8300,2,120.8300,12.5000,False,False,2025-10-31,0,0,1
2,C000003,2025-07-12,north_america,influencer,172,6.3710,170.7700,3,170.7700,6.6700,False,False,2025-12-25,0,0,1
3,C000004,2024-08-08,rest_of_world,influencer,510,5.7290,491.9300,8,491.9300,8.7500,False,False,2025-12-13,0,0,1
4,C000005,2025-07-03,uk,affiliate,181,4.0360,104.9700,2,104.9700,0.0000,True,False,2025-07-29,1,0,0


,missing_values
treatment,0
outcome,0
purchase_frequency,0
prior_spending,0
customer_tenure_days,0
customer_region,0
acquisition_channel,0
average_discount_percent,0
discount_user,0


## Identification of the Causal Estimand

Target estimand: average treatment effect on churn.

$$ATE = E[Y(1) - Y(0)]$$

Under the backdoor assumptions below, this can be identified by adjusting for observed customer characteristics:

$$E_X[E(Y \mid T=1, X) - E(Y \mid T=0, X)]$$

where `T` is price-increase exposure, `Y` is churn, and `X` contains purchase frequency, prior spending, tenure, region, and acquisition channel.

In [26]:
dowhy_graph = """
digraph {
    customer_region -> price_increase_occurred;
    customer_region -> churned;
    acquisition_channel -> price_increase_occurred;
    acquisition_channel -> churned;
    purchase_frequency -> price_increase_occurred;
    purchase_frequency -> churned;
    customer_tenure_days -> price_increase_occurred;
    customer_tenure_days -> churned;
    prior_spending -> price_increase_occurred;
    prior_spending -> churned;
    price_increase_occurred -> average_discount_percent;
    average_discount_percent -> churned;
    price_increase_occurred -> churned;
}
"""

if DOWHY_AVAILABLE:
    try:
        dowhy_model = CausalModel(
            data=customers,
            treatment="price_increase_occurred",
            outcome="churned",
            graph=dowhy_graph,
        )
        identified_estimand = dowhy_model.identify_effect(proceed_when_unidentifiable=True)
        print(identified_estimand)
    except Exception as error:
        print(f"DoWhy is installed, but identification did not complete in this environment: {error}")
else:
    print("DoWhy is not available in this environment.")
    print("Using a manual backdoor-adjustment workflow with propensity score weighting.")

DoWhy is not available in this environment.
Using a manual backdoor-adjustment workflow with propensity score weighting.


## Naive, Unadjusted Comparison

This is the same style of comparison used in the A/B test notebook. It is included here only as a baseline comparison.

In [27]:
group_summary = (
    customers.groupby("treatment")
    .agg(
        customers=("customer_id", "count"),
        churned_customers=("outcome", "sum"),
        churn_rate=("outcome", "mean"),
    )
    .rename(index={0: "Control: not exposed", 1: "Treatment: exposed"})
)

control_churn_rate = group_summary.loc["Control: not exposed", "churn_rate"]
treatment_churn_rate = group_summary.loc["Treatment: exposed", "churn_rate"]
naive_difference = treatment_churn_rate - control_churn_rate
current_regenerated_benchmark = 0.0216

display(group_summary)
print(f"Naive treatment-control churn difference: {naive_difference * 100:.2f} percentage points")
print(f"Current regenerated-data benchmark: {current_regenerated_benchmark * 100:.2f} percentage points")

,customers,churned_customers,churn_rate
treatment,,,
Control: not exposed,8929,595,0.0666
Treatment: exposed,3571,315,0.0882


Naive treatment-control churn difference: 2.16 percentage points
Current regenerated-data benchmark: 2.16 percentage points


## Backdoor Adjustment with Propensity Score Weighting

The propensity model below predicts treatment assignment, not churn. It is used to balance exposed and unexposed customers on observed pre-treatment characteristics before estimating the treatment effect.

In [28]:
def make_design_matrix(data, features):
    """Create a numeric treatment-model matrix from adjustment variables."""
    modeling_data = data[features].copy()
    for feature in features:
        if pd.api.types.is_bool_dtype(modeling_data[feature]):
            modeling_data[feature] = modeling_data[feature].astype(int)

    numeric_features = [
        feature
        for feature in features
        if pd.api.types.is_numeric_dtype(modeling_data[feature])
    ]
    for feature in numeric_features:
        standard_deviation = modeling_data[feature].std(ddof=0)
        if standard_deviation == 0 or pd.isna(standard_deviation):
            modeling_data[feature] = 0.0
        else:
            modeling_data[feature] = (
                modeling_data[feature] - modeling_data[feature].mean()
            ) / standard_deviation

    categorical_features = [feature for feature in features if feature not in numeric_features]
    design_matrix = pd.get_dummies(
        modeling_data,
        columns=categorical_features,
        drop_first=True,
        dtype=float,
    )
    return design_matrix.astype(float)


def fit_logistic_regression(
    design_matrix,
    target,
    ridge_penalty=1.0,
    max_iter=100,
    tolerance=1e-8,
):
    """Fit a small ridge-stabilized logistic model with NumPy."""
    x_matrix = np.column_stack(
        [np.ones(len(design_matrix)), design_matrix.to_numpy(dtype=float)]
    )
    target_array = np.asarray(target, dtype=float)
    coefficients = np.zeros(x_matrix.shape[1])
    penalty_matrix = np.eye(x_matrix.shape[1]) * ridge_penalty
    penalty_matrix[0, 0] = 0.0

    for _ in range(max_iter):
        linear_score = np.clip(x_matrix @ coefficients, -30, 30)
        probabilities = 1 / (1 + np.exp(-linear_score))
        weights = np.clip(probabilities * (1 - probabilities), 1e-8, None)
        gradient = x_matrix.T @ (probabilities - target_array) + penalty_matrix @ coefficients
        hessian = x_matrix.T @ (weights[:, None] * x_matrix) + penalty_matrix
        try:
            step = np.linalg.solve(hessian, gradient)
        except np.linalg.LinAlgError:
            step = np.linalg.lstsq(hessian, gradient, rcond=None)[0]
        coefficients -= step
        if np.linalg.norm(step) < tolerance:
            break

    linear_score = np.clip(x_matrix @ coefficients, -30, 30)
    probabilities = 1 / (1 + np.exp(-linear_score))
    return coefficients, probabilities


def fit_propensity_scores(data, features, lower_clip=0.02, upper_clip=0.98):
    """Estimate treatment propensity scores from observed adjustment variables."""
    design_matrix = make_design_matrix(data, features)
    coefficients, propensity_scores = fit_logistic_regression(
        design_matrix,
        data["treatment"],
    )
    propensity_model = {
        "columns": ["intercept", *design_matrix.columns.tolist()],
        "coefficients": coefficients,
    }
    return np.clip(propensity_scores, lower_clip, upper_clip), propensity_model


def estimate_ipw_ate(data, propensity_scores):
    """Estimate the ATE using normalized inverse-probability weights."""
    treatment = data["treatment"].to_numpy(dtype=float)
    outcome = data["outcome"].to_numpy(dtype=float)
    treated_weights = treatment / propensity_scores
    control_weights = (1 - treatment) / (1 - propensity_scores)
    treated_mean = np.sum(treated_weights * outcome) / np.sum(treated_weights)
    control_mean = np.sum(control_weights * outcome) / np.sum(control_weights)
    all_weights = np.where(treatment == 1, 1 / propensity_scores, 1 / (1 - propensity_scores))
    effective_sample_size = np.sum(all_weights) ** 2 / np.sum(all_weights ** 2)
    return {
        "adjusted_treated_churn": treated_mean,
        "adjusted_control_churn": control_mean,
        "ate": treated_mean - control_mean,
        "effective_sample_size": effective_sample_size,
    }


def bootstrap_ipw_ci(data, propensity_scores, n_bootstrap=500, seed=42):
    """Bootstrap the IPW estimate using fixed propensity scores."""
    rng = np.random.default_rng(seed)
    row_index = np.arange(len(data))
    bootstrap_estimates = []
    for _ in range(n_bootstrap):
        sampled_index = rng.choice(row_index, size=len(row_index), replace=True)
        sampled_data = data.iloc[sampled_index]
        sampled_scores = propensity_scores[sampled_index]
        bootstrap_estimates.append(estimate_ipw_ate(sampled_data, sampled_scores)["ate"])
    return np.percentile(bootstrap_estimates, [2.5, 97.5]), np.std(bootstrap_estimates, ddof=1)


def standardized_mean_difference(values, treatment, weights=None):
    """Calculate the treated-control standardized mean difference."""
    values = np.asarray(values, dtype=float)
    treatment = np.asarray(treatment, dtype=int)
    if weights is None:
        weights = np.ones_like(values, dtype=float)
    else:
        weights = np.asarray(weights, dtype=float)

    treated_mask = treatment == 1
    control_mask = treatment == 0
    treated_values = values[treated_mask]
    control_values = values[control_mask]
    treated_weights = weights[treated_mask]
    control_weights = weights[control_mask]

    treated_mean = np.average(treated_values, weights=treated_weights)
    control_mean = np.average(control_values, weights=control_weights)
    treated_var = np.average((treated_values - treated_mean) ** 2, weights=treated_weights)
    control_var = np.average((control_values - control_mean) ** 2, weights=control_weights)
    pooled_std = sqrt((treated_var + control_var) / 2)
    if pooled_std == 0:
        return 0.0
    return (treated_mean - control_mean) / pooled_std


def make_balance_table(data, features, propensity_scores):
    """Compare covariate balance before and after propensity weighting."""
    treatment = data["treatment"].to_numpy()
    weights = np.where(treatment == 1, 1 / propensity_scores, 1 / (1 - propensity_scores))
    rows = []

    for feature in features:
        if pd.api.types.is_numeric_dtype(data[feature]):
            feature_values = data[feature]
            rows.append(
                {
                    "variable": feature,
                    "smd_before": standardized_mean_difference(feature_values, treatment),
                    "smd_after": standardized_mean_difference(feature_values, treatment, weights),
                }
            )
        else:
            for category in sorted(data[feature].dropna().unique()):
                feature_values = (data[feature] == category).astype(int)
                rows.append(
                    {
                        "variable": f"{feature}={category}",
                        "smd_before": standardized_mean_difference(feature_values, treatment),
                        "smd_after": standardized_mean_difference(feature_values, treatment, weights),
                    }
                )
    balance = pd.DataFrame(rows)
    balance["abs_smd_before"] = balance["smd_before"].abs()
    balance["abs_smd_after"] = balance["smd_after"].abs()
    return balance.sort_values("abs_smd_before", ascending=False)

In [29]:
primary_propensity_scores, primary_propensity_model = fit_propensity_scores(
    customers,
    primary_confounders,
)
primary_estimate = estimate_ipw_ate(customers, primary_propensity_scores)
primary_ci, primary_bootstrap_se = bootstrap_ipw_ci(
    customers,
    primary_propensity_scores,
    n_bootstrap=500,
)

primary_result = pd.DataFrame(
    {
        "metric": [
            "adjusted treated churn",
            "adjusted control churn",
            "adjusted ATE",
            "bootstrap CI lower",
            "bootstrap CI upper",
            "bootstrap standard error",
            "effective sample size",
        ],
        "value": [
            primary_estimate["adjusted_treated_churn"],
            primary_estimate["adjusted_control_churn"],
            primary_estimate["ate"],
            primary_ci[0],
            primary_ci[1],
            primary_bootstrap_se,
            primary_estimate["effective_sample_size"],
        ],
    }
)

display(primary_result)
print(f"Primary adjusted ATE: {primary_estimate['ate'] * 100:.2f} percentage points")

,metric,value
0,adjusted treated churn,0.0896
1,adjusted control churn,0.0651
2,adjusted ATE,0.0245
3,bootstrap CI lower,0.0131
4,bootstrap CI upper,0.0369
5,bootstrap standard error,0.0059
6,effective sample size,"8,873.7767"


Primary adjusted ATE: 2.45 percentage points


## Propensity Overlap and Balance Check

In [30]:
propensity_summary = pd.DataFrame(
    {
        "treatment_group": customers["treatment"].map({0: "Control", 1: "Treatment"}),
        "propensity_score": primary_propensity_scores,
    }
).groupby("treatment_group")["propensity_score"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
)

overlap_checks = pd.DataFrame(
    {
        "metric": [
            "minimum propensity score",
            "maximum propensity score",
            "share below 0.05",
            "share above 0.95",
        ],
        "value": [
            primary_propensity_scores.min(),
            primary_propensity_scores.max(),
            (primary_propensity_scores < 0.05).mean(),
            (primary_propensity_scores > 0.95).mean(),
        ],
    }
)

balance_table = make_balance_table(customers, primary_confounders, primary_propensity_scores)

display(propensity_summary)
display(overlap_checks)
display(balance_table.head(12))

,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
treatment_group,,,,,,,,,,,,
Control,"8,929.0000",0.2621,0.1151,0.1106,0.1209,0.1293,0.1766,0.2350,0.3195,0.4912,0.6321,0.8931
Treatment,"3,571.0000",0.3446,0.1465,0.1118,0.1278,0.1452,0.2321,0.3183,0.4313,0.6243,0.7499,0.8712


,metric,value
0,minimum propensity score,0.1106
1,maximum propensity score,0.8931
2,share below 0.05,0.0000
3,share above 0.95,0.0000


,variable,smd_before,smd_after,abs_smd_before,abs_smd_after
1,prior_spending,0.6045,0.0079,0.6045,0.0079
2,customer_tenure_days,0.3993,0.0088,0.3993,0.0088
0,purchase_frequency,0.2775,0.0006,0.2775,0.0006
12,acquisition_channel=organic_search,0.0370,0.0096,0.0370,0.0096
10,acquisition_channel=email,-0.0348,-0.0004,0.0348,0.0004
13,acquisition_channel=paid_social,-0.0335,-0.0023,0.0335,0.0023
11,acquisition_channel=influencer,0.0195,0.0011,0.0195,0.0011
4,customer_region=europe,0.0163,0.0023,0.0163,0.0023
6,customer_region=rest_of_world,-0.0092,-0.0037,0.0092,0.0037
8,acquisition_channel=affiliate,0.0068,0.0027,0.0068,0.0027


## Discount-Adjusted Sensitivity Estimate

This version adds average discount usage to the adjustment set. Because discounts may occur after price exposure, this is interpreted as a sensitivity analysis rather than the primary total-effect estimate.

In [31]:
discount_propensity_scores, discount_propensity_model = fit_propensity_scores(
    customers,
    discount_sensitivity_confounders,
)
discount_adjusted_estimate = estimate_ipw_ate(customers, discount_propensity_scores)

sensitivity_results = pd.DataFrame(
    {
        "estimate": ["primary pre-treatment adjustment", "discount-adjusted sensitivity"],
        "adjusted_treated_churn": [
            primary_estimate["adjusted_treated_churn"],
            discount_adjusted_estimate["adjusted_treated_churn"],
        ],
        "adjusted_control_churn": [
            primary_estimate["adjusted_control_churn"],
            discount_adjusted_estimate["adjusted_control_churn"],
        ],
        "ate": [primary_estimate["ate"], discount_adjusted_estimate["ate"]],
        "ate_percentage_points": [
            primary_estimate["ate"] * 100,
            discount_adjusted_estimate["ate"] * 100,
        ],
    }
)

display(sensitivity_results)

,estimate,adjusted_treated_churn,adjusted_control_churn,ate,ate_percentage_points
0,primary pre-treatment adjustment,0.0896,0.0651,0.0245,2.4542
1,discount-adjusted sensitivity,0.0906,0.0650,0.0255,2.5541


## Robustness / Refutation Check

A random common-cause refuter adds a synthetic noise variable to the adjustment set. A stable estimate should not change meaningfully when this irrelevant variable is included.

In [32]:
rng = np.random.default_rng(42)
refuter_data = customers.copy()
refuter_data["random_common_cause"] = rng.normal(size=len(refuter_data))
refuter_scores, refuter_model = fit_propensity_scores(
    refuter_data,
    primary_confounders + ["random_common_cause"],
)
refuter_estimate = estimate_ipw_ate(refuter_data, refuter_scores)
refuter_change = refuter_estimate["ate"] - primary_estimate["ate"]

refuter_results = pd.DataFrame(
    {
        "metric": [
            "primary adjusted ATE",
            "ATE after random common cause",
            "change from primary ATE",
        ],
        "value": [
            primary_estimate["ate"],
            refuter_estimate["ate"],
            refuter_change,
        ],
        "percentage_points": [
            primary_estimate["ate"] * 100,
            refuter_estimate["ate"] * 100,
            refuter_change * 100,
        ],
    }
)

display(refuter_results)

,metric,value,percentage_points
0,primary adjusted ATE,0.0245,2.4542
1,ATE after random common cause,0.0244,2.4393
2,change from primary ATE,-0.0001,-0.0149


## Assumptions and Limitations

- Conditional exchangeability: after adjusting for observed customer characteristics, exposed and unexposed customers are assumed comparable on churn risk.
- Positivity: each type of customer has a nonzero chance of being exposed and unexposed to a price increase.
- Consistency: each customer's observed churn outcome corresponds to the exposure condition assigned in the data.
- No interference: one customer's exposure does not affect another customer's churn outcome.
- Correct adjustment set: region, acquisition channel, purchase frequency, tenure, and prior spending are assumed to block the main backdoor paths.
- Discount usage is ambiguous: it may reflect targeting, customer behavior, or post-treatment mitigation, so it is handled as a sensitivity adjustment rather than the primary total-effect adjustment.
- This is synthetic data. The estimate validates the project workflow, not a real Gymshark business effect.

## Causal Inference Findings

In [33]:
adjusted_effect_pp = primary_estimate["ate"] * 100
naive_effect_pp = naive_difference * 100
displayed_adjusted_effect_pp = round(adjusted_effect_pp, 2)
displayed_naive_effect_pp = round(naive_effect_pp, 2)
difference_from_naive_pp = displayed_adjusted_effect_pp - displayed_naive_effect_pp
ci_lower_pp = primary_ci[0] * 100
ci_upper_pp = primary_ci[1] * 100
discount_sensitivity_pp = discount_adjusted_estimate["ate"] * 100
refuter_change_pp = refuter_change * 100

if adjusted_effect_pp > naive_effect_pp:
    comparison_text = "larger than"
elif adjusted_effect_pp < naive_effect_pp:
    comparison_text = "smaller than"
else:
    comparison_text = "equal to"

print("Causal Inference Findings")
print(f"- Naive churn difference: {naive_effect_pp:.2f} percentage points higher churn among price-exposed customers.")
print(f"- Primary adjusted IPW estimate: {adjusted_effect_pp:.2f} percentage points, with bootstrap 95% CI [{ci_lower_pp:.2f}, {ci_upper_pp:.2f}] percentage points.")
print(f"- The adjusted estimate is {abs(difference_from_naive_pp):.2f} percentage points {comparison_text} the unadjusted {displayed_naive_effect_pp:.2f} percentage-point result.")
print(f"- The discount-adjusted sensitivity estimate is {discount_sensitivity_pp:.2f} percentage points, which is close to the primary estimate.")
print(f"- Adding a random common cause changed the estimate by {refuter_change_pp:.2f} percentage points, suggesting the result is stable to this simple refutation check.")
print("- Interpretation: under the stated backdoor-adjustment assumptions, price-increase exposure is associated with a higher estimated churn risk in the synthetic data. This is a causal workflow estimate, not a churn prediction model and not evidence about a real company.")

Causal Inference Findings
- Naive churn difference: 2.16 percentage points higher churn among price-exposed customers.
- Primary adjusted IPW estimate: 2.45 percentage points, with bootstrap 95% CI [1.31, 3.69] percentage points.
- The adjusted estimate is 0.29 percentage points larger than the unadjusted 2.16 percentage-point result.
- The discount-adjusted sensitivity estimate is 2.55 percentage points, which is close to the primary estimate.
- Adding a random common cause changed the estimate by -0.01 percentage points, suggesting the result is stable to this simple refutation check.
- Interpretation: under the stated backdoor-adjustment assumptions, price-increase exposure is associated with a higher estimated churn risk in the synthetic data. This is a causal workflow estimate, not a churn prediction model and not evidence about a real company.
